# Install Library

In [ ]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

# Import Library

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from google.colab import drive

# Connect to Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path folder tempat file CSV berada
folder_path = "/content/drive/MyDrive/File/CODES/Summarize/1_Summary_Detik_Berita_50_Base1/Sum_Detik_Berita_50_Base1"

# Mengambil semua file dengan ekstensi .csv dalam folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith(".csv")]

# Membaca dan menggabungkan semua file
dataframes = []
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Menggabungkan semua DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

# # Menyimpan hasil penggabungan ke file baru
# output_path = os.path.join(folder_path, "/content/drive/MyDrive/0.SKRIPSI/CODES/Similarity/1_Similarity_Detik_Berita_Base1/detikberita_sum_50_base1_concat.csv")
# combined_df.to_csv(output_path, index=False)


In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13249 entries, 0 to 13248
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         13249 non-null  object
 1   category      13249 non-null  object
 2   publish_date  13249 non-null  object
 3   author        13249 non-null  object
 4   article_url   13249 non-null  object
 5   content       13249 non-null  object
 6   summary       13249 non-null  object
dtypes: object(7)
memory usage: 724.7+ KB


In [ ]:
combined_df.head()

,title,category,publish_date,author,article_url,content,summary
0,Prabowo Sarankan Otorita Prioritaskan Bangun G...,detikNews,2024-08-12,Eva Safitri,https://news.detik.com/berita/d-7486738/prabow...,Menhan sekaligus Presiden terpilih Prabowo Sub...,Menhan sekaligus Presiden terpilih Prabowo Sub...
1,Warga Sebut Mobil Nyangkut di Separator Margon...,detikNews,2024-08-12,Devi Puspitasari,https://news.detik.com/berita/d-7486696/warga-...,Mobil mengalami kecelakaan hingga tersangkut d...,Mobil mengalami kecelakaan hingga tersangkut d...
2,Penyidikan Kasus Surya Darmadi Dihentikan KPK ...,detikNews,2024-08-12,Adrial akbar,https://news.detik.com/berita/d-7486641/penyid...,Komisi Pemberantasan Korupsi (KPK) telah mener...,Komisi Pemberantasan Korupsi (KPK) telah mener...
3,Horor Korban Jambret Terseret Motor Dikira Rib...,detikNews,2024-08-12,Muchamad Sholihin,https://news.detik.com/berita/d-7486614/horor-...,Peristiwa horor dialami seorang wanita bernama...,Peristiwa horor dialami seorang wanita bernama...
4,KPK Periksa Eks Anggota DPR Miryam S Haryani T...,detikNews,2024-08-12,Adrial akbar,https://news.detik.com/berita/d-7486362/kpk-pe...,KPK menjadwalkan pemeriksaan mantan anggota DP...,KPK menjadwalkan pemeriksaan mantan anggota DP...


In [ ]:
nan_summary = combined_df[combined_df['summary'].isna()]


nan_summary

,title,category,publish_date,author,article_url,content,summary


In [ ]:
combined_df=combined_df.dropna()

# Duplicate


In [ ]:
data = combined_df.copy()

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13249 entries, 0 to 13248
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         13249 non-null  object
 1   category      13249 non-null  object
 2   publish_date  13249 non-null  object
 3   author        13249 non-null  object
 4   article_url   13249 non-null  object
 5   content       13249 non-null  object
 6   summary       13249 non-null  object
dtypes: object(7)
memory usage: 724.7+ KB


# Tokenize

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p2", use_fast=True)
# tokenizer.add_tokens([ 'NaN' ])

In [ ]:
# def tokenize_function(text):
#     return tokenizer(text, padding='max_length', max_length=256, truncation=True)

# # Tokenisasi kolom 'title' dan 'summary'
# data['tokenized_title'] = data['title'].apply(lambda x: tokenize_function(x))
# data['tokenized_summary'] = data['summary'].apply(lambda x: tokenize_function(x))

In [ ]:
# data.head()

In [ ]:
# # Menampilkan hasil tokenisasi untuk 'tokenized_title' dan 'tokenized_content'
# print("Tokenized Title:")
# print(data[['title', 'tokenized_title']])

# print("\nTokenized summary:")
# print(data[['summary', 'tokenized_summary']])

# # Menampilkan hanya input_ids yang lebih mudah dibaca
# print("\nInput IDs for Tokenized Title:")
# print(data['tokenized_title'].apply(lambda x: x['input_ids']))

# print("\nInput IDs for Tokenized summary:")
# print(data['tokenized_summary'].apply(lambda x: x['input_ids']))

Mencoba kode lain agar tokenize lebih mudah di pahami

# Word Embedding

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Memuat IndoBERT dan tokenizer
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1", use_fast=True)
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [ ]:
# Fungsi untuk mendapatkan embedding dari IndoBERT
def get_embeddings(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    # Mengambil hidden states dari layer terakhir dan rata-rata embeddingsnya
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk mengonversi tokenized data ke dalam bentuk input_ids dan mendapatkan embedding
def get_input_ids_and_embeddings(data_column):
    input_ids_list = []
    embeddings_list = []

    for text in data_column:
        # Tokenisasi
        tokens = tokenizer(text, padding='max_length', max_length=256, truncation=True, return_tensors="pt")
        input_ids = tokens['input_ids']  # Ambil input_ids
        input_ids_list.append(input_ids)

        # Mendapatkan embeddings
        embedding = get_embeddings(input_ids)
        embeddings_list.append(embedding)

    return input_ids_list, embeddings_list

# Dapatkan input_ids dan embeddings untuk title dan summary
title_input_ids, title_embeddings = get_input_ids_and_embeddings(data['title'])
summary_input_ids, summary_embeddings = get_input_ids_and_embeddings(data['summary'])


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

## Mengubah embeddings menjadi array numpy

In [ ]:
# Mengubah embeddings menjadi array numpy agar bisa disimpan dalam DataFrame
title_embeddings_array = np.vstack([embedding.numpy() for embedding in title_embeddings])
summary_embeddings_array = np.vstack([embedding.numpy() for embedding in summary_embeddings])

# Membuat DataFrame baru dengan embeddings numerik
data_embeddings = pd.DataFrame({
    'title_embeddings': list(title_embeddings_array),
    'summary_embeddings': list(summary_embeddings_array)
})

# Menyimpan DataFrame embeddings ke dalam file CSV (atau format lain)
# data_embeddings.to_csv('/content/drive/MyDrive/File/CODES/Similarity/Detik Berita/detik_berita_embeddings.csv', index=False)


# Cosine Similarity

In [ ]:
# hitung cosine similarity pada matrix berita
cosine_sim = cosine_similarity(title_embeddings_array, summary_embeddings_array)
print(cosine_sim)

[[0.39743182 0.08263262 0.16272368 ... 0.88219666 0.4129303  0.8092322 ]
 [0.40136445 0.08776305 0.16605133 ... 0.8857981  0.41709837 0.8138304 ]
 [0.39535975 0.08522928 0.16942966 ... 0.8800826  0.41067275 0.80712414]
 ...
 [0.40607548 0.08772117 0.1685186  ... 0.8937255  0.42193395 0.82156384]
 [0.3931021  0.08393773 0.16293524 ... 0.87755513 0.41084927 0.80450904]
 [0.39085555 0.08555878 0.16440985 ... 0.8766229  0.40758425 0.8029333 ]]


In [ ]:
# membuat dataframe dari varible cosine similarity dengan baris dan kolom title
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['summary'], columns=data['title'])

print(cosine_sim_df.shape)

(13249, 13249)


## Check Similarity Result

In [ ]:
# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # The index is a summary, and you need to find the corresponding title
    # to access the similarity value. However, since you have both summaries
    # and titles as indices, you can access the similarity directly using
    # the index and the corresponding title from the 'data' DataFrame.

    title = data.loc[data['summary'] == index, 'title'].iloc[0] # Get the title for the current summary
    similarity_value = cosine_sim_df.loc[index, title]  # Access the similarity using both summary and title

    print(f"Title: {title}")
    print(f"Summary: {index}")
    print(f"Cosine Similarity: {similarity_value}\n")
    print("-" * 50)  # Pemisah antar pasangan

Streaming output truncated to the last 5000 lines.
Summary: Lomba 17 Agustus merupakan kegiatan wajib untuk memeriahkan peringatan HUT RI atau Hari Kemerdekaan RI. Berikut ini rekomendasi lomba 17 Agustus untuk anak-anak hingga orang dewasa. Sejumlah perlombaan bisa dijadikan pilihan saat momentum peringatan 17 Agustus. Perlombaan ini bisa dilakukan secara tim atau individu. Peserta memindahkan bendera dari satu tempat ke tempat lainnya. Dalam lomba ini, sebuah tim yang terdiri dari dua hingga empat orang harus berjoget sambil meletakkan balon di kepala atau perut mereka. Balon dijaga tanpa disentuh dan pastikan tidak jatuh. Perlombaan ini membutuhkan kelompok yang terdiri dari 4 sampai 6 orang. Apabila karet jatuh, harus diulang dari peserta pertama. Kelompok yang berhasil memindahkan karet dengan jumlah terbanyak, jadi pemenang. Perlombaan kelereng membutuhkan alat berupa sendok dan satu buah kelereng. Bakiak adalah permainan tradisional yang menggunakan sandal kayu panjang. Permaina

In [ ]:
# List to store the results
results = []

# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # Mendapatkan judul yang sesuai dengan summary berdasarkan index
    title = data.loc[data['summary'] == index, 'title'].iloc[0]
    similarity_value = cosine_sim_df.loc[index, title]  # Mengakses nilai similarity

    # Menyimpan pasangan dan nilai similarity ke dalam list
    results.append([title, index, similarity_value])

# Membuat DataFrame untuk hasilnya
similarity_df = pd.DataFrame(results, columns=['Title', 'Summary', 'Cosine Similarity'])


In [ ]:
similarity_df.head()

,Title,Summary,Cosine Similarity
0,Prabowo Sarankan Otorita Prioritaskan Bangun G...,Menhan sekaligus Presiden terpilih Prabowo Sub...,0.397432
1,Warga Sebut Mobil Nyangkut di Separator Margon...,Mobil mengalami kecelakaan hingga tersangkut d...,0.087763
2,Penyidikan Kasus Surya Darmadi Dihentikan KPK ...,Komisi Pemberantasan Korupsi (KPK) telah mener...,0.16943
3,Horor Korban Jambret Terseret Motor Dikira Rib...,Peristiwa horor dialami seorang wanita bernama...,0.122731
4,KPK Periksa Eks Anggota DPR Miryam S Haryani T...,KPK menjadwalkan pemeriksaan mantan anggota DP...,0.160098


## Save Similarity

In [ ]:
# Menyimpan DataFrame ke file CSV di lokasi yang ditentukan
similarity_df.to_csv('/content/drive/MyDrive/File/CODES/Similarity/1_Similarity_Detik_Berita_Base1/similarity_detik_berita_base1.csv', index=False)